# LoRA マージ + AWQ 量子化

本ノートブックは LoRA アダプタをベースモデルへマージし、llmcompressor ベースの AWQ 量子化を行う手順を機能別にまとめたものです。

1. Google Drive のマウントと依存パッケージのインストール
2. 入出力パスや量子化パラメータの設定（ランタイム再起動後も自動復元）
3. LoRA マージの実行（スキップ可）
4. AWQ 量子化の実行（LoRA なしでも利用可）

再起動が発生した場合はセル1→2を順に再実行することで設定が復元されます。


In [ ]:
# === 1. Colab utility setup ===
import os
import shlex
import sys
from typing import Sequence

try:
    from IPython import get_ipython
except ImportError:  # pragma: no cover
    get_ipython = None  # type: ignore


def _require_ipython():
    ip = get_ipython() if callable(get_ipython) else None
    if ip is None:
        raise RuntimeError("IPython 上で実行してください。Google Colab を想定しています。")
    return ip


def clone_repo(url: str, target: str, branch: str | None = None) -> None:
    if os.path.exists(target):
        print("既存のリポジトリを再利用します:", target)
        return
    ip = _require_ipython()
    if branch:
        print(f"リポジトリを clone します: {url} (branch={branch})")
        ip.system(f"git clone --branch {branch} --single-branch {url} {target}")
    else:
        print("リポジトリを clone します:", url)
        ip.system(f"git clone {url} {target}")


def _run_pip(arguments: Sequence[str]) -> None:
    ip = _require_ipython()
    command = " ".join(shlex.quote(str(arg)) for arg in arguments)
    print("pip", command)
    ip.run_line_magic("pip", command)


def pip_install(
    packages: Sequence[str],
    *,
    index_url: str | None = None,
    extra_index_url: str | None = None,
    upgrade: bool = True,
    extra_args: Sequence[str] | None = None,
) -> None:
    items = [pkg for pkg in packages if pkg]
    if not items:
        return
    args = ["install"]
    if upgrade:
        args.append("--upgrade")
    if index_url:
        args.extend(["--index-url", index_url])
    if extra_index_url:
        args.extend(["--extra-index-url", extra_index_url])
    if extra_args:
        args.extend(extra_args)
    args.extend(items)
    _run_pip(args)


def pip_uninstall(packages: Sequence[str]) -> None:
    items = [pkg for pkg in packages if pkg]
    if not items:
        return
    args = ["uninstall", "-y"]
    args.extend(items)
    _run_pip(args)


# === 0. Dependency refresh ===
%pip install -U "llmcompressor>=0.8.1" "compressed-tensors>=0.12.1"
%pip install -U "vllm>=0.11.1"


REPO_URL = "https://github.com/fouga1221/llm-lab2.git"
DEFAULT_REPO_DIR = "/content/llm-lab2" if "google.colab" in sys.modules else os.path.abspath("..")
REPO_DIR = DEFAULT_REPO_DIR
REPO_BRANCH = "main2"  # 必要に応じて変更

# sys.path へリポジトリルートを追加（重複は避ける）
repo_path_for_import = os.path.abspath(REPO_DIR)
if repo_path_for_import not in sys.path:
    sys.path.insert(0, repo_path_for_import)


IS_COLAB = "google.colab" in sys.modules
DEFAULT_TORCH_INDEX = "https://download.pytorch.org/whl/cu126" if IS_COLAB else None
TORCH_INDEX_URL = os.environ.get("PYTORCH_WHL_INDEX_URL", DEFAULT_TORCH_INDEX)
TORCH_EXTRA_INDEX_URL = os.environ.get("PYTORCH_EXTRA_INDEX_URL", "https://pypi.org/simple")

BASE_BOOTSTRAP = [
    "pip>=24.2",
    "setuptools==79.0.1",
    "wheel>=0.44.0",
    "packaging>=24.2",
    "jedi>=0.19.1",
]

TORCH_PACKAGES = [
    "torch==2.8.0",
    "torchvision==0.23.0",
    "torchaudio==2.8.0",
]

AWQ_NUMPY_SPEC = ["numpy==2.1.3"]
AWQ_RUNTIME_PACKAGES = [
    "transformers==4.56.2",
    "peft==0.17.1",
    "accelerate==1.10.1",
    "safetensors>=0.4.2",
]
AWQ_DATA_PACKAGES = [
    "datasets==4.0.0",
    "pyarrow==19.0.1",
]
AWQ_TELEMETRY_PACKAGES = [
    "opentelemetry-api==1.37.0",
    "opentelemetry-sdk==1.37.0",
    "opentelemetry-exporter-otlp==1.37.0",
    "opentelemetry-exporter-otlp-proto-grpc==1.37.0",
    "opentelemetry-exporter-otlp-proto-http==1.37.0",
    "opentelemetry-exporter-otlp-proto-common==1.37.0",
    "opentelemetry-proto==1.37.0",
    "opentelemetry-semantic-conventions==0.58b0",
]
AWQ_LLMCOMPRESSOR_PACKAGES = [
    "llmcompressor==0.8.1",
    "compressed-tensors==0.12.2",
]

EXTRA_PACKAGES: Sequence[str] = []

clone_repo(REPO_URL, REPO_DIR, branch=REPO_BRANCH)
pip_install(BASE_BOOTSTRAP)
pip_install(TORCH_PACKAGES, index_url=TORCH_INDEX_URL, extra_index_url=TORCH_EXTRA_INDEX_URL)
pip_install(AWQ_NUMPY_SPEC)
pip_install(AWQ_RUNTIME_PACKAGES)
pip_install(AWQ_DATA_PACKAGES)
pip_install(AWQ_TELEMETRY_PACKAGES)
pip_install(AWQ_LLMCOMPRESSOR_PACKAGES)
if EXTRA_PACKAGES:
    pip_install(EXTRA_PACKAGES)

print("依存パッケージのインストールが完了しました。ランタイム再起動時はセル1と2を再実行してください。")


In [ ]:

# === 2. パスとベースモデル準備（必須） ===
import json
import os
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules

COLAB_DATA_ROOT_DEFAULT = Path("/content/drive/MyDrive/ProjectForte/llm-lab-save")
COLAB_WORKSPACE_ROOT_DEFAULT = Path("/content/llm-lab-save")
COLAB_DATA_ROOT = Path(os.environ.get("LLMLAB_DRIVE_DATA_ROOT", COLAB_DATA_ROOT_DEFAULT))
workspace_override = os.environ.get("LLMLAB_WORKSPACE_DATA_ROOT")
COLAB_WORKSPACE_ROOT = Path(workspace_override) if workspace_override else COLAB_WORKSPACE_ROOT_DEFAULT

if IS_COLAB:
    from google.colab import drive  # type: ignore

    skip_mount = os.environ.get("LLMLAB_SKIP_DRIVE_MOUNT", "").lower() in {"1", "true", "yes"}
    if skip_mount:
        print("環境変数 LLMLAB_SKIP_DRIVE_MOUNT により Google Drive マウントをスキップします。")
    else:
        drive.mount("/content/drive", force_remount=False)

    if workspace_override:
        COLAB_WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)
        DATA_ROOT = COLAB_WORKSPACE_ROOT
    else:
        COLAB_DATA_ROOT.mkdir(parents=True, exist_ok=True)
        if COLAB_WORKSPACE_ROOT.exists() or COLAB_WORKSPACE_ROOT.is_symlink():
            print("既存のワークスペースを再利用します:", COLAB_WORKSPACE_ROOT)
        else:
            COLAB_WORKSPACE_ROOT.symlink_to(COLAB_DATA_ROOT, target_is_directory=True)
        DATA_ROOT = COLAB_WORKSPACE_ROOT
else:
    local_override = os.environ.get("LLMLAB_LOCAL_DATA_ROOT")
    DATA_ROOT = Path(local_override) if local_override else Path.cwd() / "save"
    DATA_ROOT.mkdir(parents=True, exist_ok=True)

REPO_ROOT = Path(REPO_DIR).resolve()
STATE_DIR = DATA_ROOT / "runtime_state"
STATE_DIR.mkdir(parents=True, exist_ok=True)
STATE_FILE = STATE_DIR / "quantization_awq_lora.json"

DEFAULT_CONFIG = {
    "BASE_MODEL_NAME": "Qwen/Qwen3-14B",
    "LORA_DIR": str(DATA_ROOT / "models" / "adapters" / "qwen3-14b-lora-ojousama"),
    "MERGED_OUTPUT_DIR": str(DATA_ROOT / "artifacts" / "merged_qwen3_14b_ojousama"),
    "AWQ_OUTPUT_DIR": str(DATA_ROOT / "artifacts" / "awq_qwen3_14b_ojousama"),
    "EXTERNAL_MERGED_DIR": None,
    "EXTERNAL_AWQ_DIR": None,
    "ENABLE_LORA_MERGE": True,
    "ENABLE_AWQ_QUANT": True,
    "MERGE_DEVICE": "cpu",
    "AWQ_RECIPE": {
        "num_bits": 4,
        "group_size": 128,
        "max_seq_len": 2048,
        "zero_point": True,
    },
    "AWQ_CALIBRATION_SAMPLES": [
        "You are a helpful assistant. Please follow the instructions strictly.",
        "Summarise the following enterprise deck in roughly 200 Japanese characters.",
        "Explain the recovery flow for AWQ quantisation failures and list three fallback steps.",
    ],
    "AWQ_CALIBRATION_DATASET": None,
    "AWQ_CALIBRATION_DATASET_SPLIT": "train",
    "AWQ_TEXT_FIELD": "text",
}


def _deep_merge(base: dict, override: dict) -> dict:
    result = dict(base)
    for key, value in override.items():
        if key in result and isinstance(result[key], dict) and isinstance(value, dict):
            result[key] = _deep_merge(result[key], value)
        else:
            result[key] = value
    return result


def load_state(path: Path, default: dict) -> dict:
    if not path.exists():
        return dict(default)
    try:
        payload = json.loads(path.read_text(encoding="utf-8"))
        print(f"保存済みの設定を読み込みました: {path}")
        return _deep_merge(default, payload)
    except Exception as exc:
        print(f"設定ファイルの読み込みに失敗したためデフォルトを使用します: {exc}")
        return dict(default)


def persist_config(config: dict) -> None:
    STATE_FILE.write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"設定を保存しました: {STATE_FILE}")


def resolve_source_model_path(config: dict) -> str:
    explicit = config.get("SOURCE_MODEL_PATH")
    if explicit:
        return str(explicit)
    candidate_keys = [
        "EXTERNAL_MERGED_DIR",
        "LAST_MERGED_DIR",
        "MERGED_OUTPUT_DIR",
    ]
    for key in candidate_keys:
        candidate = config.get(key)
        if not candidate:
            continue
        candidate_path = Path(str(candidate))
        if candidate_path.is_dir() and any(candidate_path.iterdir()):
            return str(candidate_path)
    return config["BASE_MODEL_NAME"]


CONFIG = load_state(STATE_FILE, DEFAULT_CONFIG)
CONFIG["LORA_DIR"] = str(Path(CONFIG["LORA_DIR"]))
CONFIG["MERGED_OUTPUT_DIR"] = str(Path(CONFIG["MERGED_OUTPUT_DIR"]))
CONFIG["AWQ_OUTPUT_DIR"] = str(Path(CONFIG["AWQ_OUTPUT_DIR"]))
if CONFIG.get("EXTERNAL_MERGED_DIR"):
    CONFIG["EXTERNAL_MERGED_DIR"] = str(Path(CONFIG["EXTERNAL_MERGED_DIR"]))
if CONFIG.get("EXTERNAL_AWQ_DIR"):
    CONFIG["EXTERNAL_AWQ_DIR"] = str(Path(CONFIG["EXTERNAL_AWQ_DIR"]))

Path(CONFIG["MERGED_OUTPUT_DIR"]).mkdir(parents=True, exist_ok=True)
Path(CONFIG["AWQ_OUTPUT_DIR"]).mkdir(parents=True, exist_ok=True)

SOURCE_MODEL_PATH = resolve_source_model_path(CONFIG)
CONFIG["SOURCE_MODEL_PATH"] = SOURCE_MODEL_PATH
persist_config(CONFIG)

print(f"REPO_ROOT: {REPO_ROOT}")
print(f"DATA_ROOT: {DATA_ROOT}")
print(f"MERGED_OUTPUT_DIR: {CONFIG['MERGED_OUTPUT_DIR']}")
print(f"AWQ_OUTPUT_DIR: {CONFIG['AWQ_OUTPUT_DIR']}")
print(f"SOURCE_MODEL_PATH: {SOURCE_MODEL_PATH}")
print("LoRA マージを行わない場合はこのベースモデルを直接 AWQ に渡します。")


In [ ]:

# === 3. LoRA マージ（オプション） ===
import gc
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model_name = CONFIG["BASE_MODEL_NAME"]
lora_dir = Path(CONFIG["LORA_DIR"])
merged_dir = Path(CONFIG["MERGED_OUTPUT_DIR"])
external_merged = Path(CONFIG["EXTERNAL_MERGED_DIR"]) if CONFIG.get("EXTERNAL_MERGED_DIR") else None

SOURCE_MODEL_PATH = CONFIG.get("SOURCE_MODEL_PATH") or base_model_name

def _cleanup_cuda_cache() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def _update_source_model_path(path: str) -> None:
    CONFIG["SOURCE_MODEL_PATH"] = path
    persist_config(CONFIG)


if CONFIG.get("ENABLE_LORA_MERGE", True):
    if not lora_dir.exists():
        raise FileNotFoundError(f"LoRA ディレクトリが見つかりません: {lora_dir}")
    merge_device = str(CONFIG.get("MERGE_DEVICE", "cpu"))
    if merge_device == "cpu":
        device_map = {"": "cpu"}
        torch_dtype = torch.float32
    elif merge_device == "auto":
        device_map = "auto"
        torch_dtype = torch.float16
    else:
        device_map = {"": merge_device}
        torch_dtype = torch.float16
    print("LoRA ベースモデルをマージしています...")
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        device_map=device_map,
        torch_dtype=torch_dtype,
        trust_remote_code=True,
        low_cpu_mem_usage=False,
    )
    lora_model = PeftModel.from_pretrained(base_model, str(lora_dir))
    merged_model = lora_model.merge_and_unload()
    merged_model.save_pretrained(merged_dir, safe_tensors=True)
    tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
    tokenizer.save_pretrained(merged_dir)
    del base_model, lora_model, merged_model, tokenizer
    _cleanup_cuda_cache()
    SOURCE_MODEL_PATH = str(merged_dir)
    CONFIG["LAST_MERGED_DIR"] = SOURCE_MODEL_PATH
    _update_source_model_path(SOURCE_MODEL_PATH)
else:
    candidates = [external_merged, merged_dir]
    for candidate in candidates:
        if candidate and candidate.exists() and candidate.is_dir() and any(candidate.iterdir()):
            SOURCE_MODEL_PATH = str(candidate)
            CONFIG["LAST_MERGED_DIR"] = SOURCE_MODEL_PATH
            print("既存のマージ済みモデルを使用します:", candidate)
            break
    else:
        SOURCE_MODEL_PATH = base_model_name
        print("マージ済みモデルが見つからないためベースモデルを利用します。")
    _update_source_model_path(SOURCE_MODEL_PATH)

print("AWQ 入力モデルパス:", SOURCE_MODEL_PATH)


In [ ]:
# === 4. AWQ 量子化 ===
import json
import tempfile
from pathlib import Path
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer
from llmcompressor.modifiers.awq import AWQModifier
try:
    from llmcompressor import oneshot as run_oneshot
except ImportError:  # pragma: no cover
    from llmcompressor.entrypoints.oneshot import oneshot as run_oneshot  # type: ignore

from src.llmlab.utils.awq import prepare_awq_calib_data

SOURCE_MODEL_PATH = globals().get("SOURCE_MODEL_PATH") or CONFIG.get("SOURCE_MODEL_PATH")
if not SOURCE_MODEL_PATH:
    SOURCE_MODEL_PATH = CONFIG["BASE_MODEL_NAME"]
    CONFIG["SOURCE_MODEL_PATH"] = SOURCE_MODEL_PATH
    persist_config(CONFIG)
    print("LoRA マージをスキップしたためベースモデルを AWQ に渡します。")

if not CONFIG.get("ENABLE_AWQ_QUANT", True):
    print("CONFIG['ENABLE_AWQ_QUANT'] が False のため量子化をスキップします。")
    AWQ_MODEL_PATH = SOURCE_MODEL_PATH
else:
    target_dir = Path(CONFIG.get("EXTERNAL_AWQ_DIR") or CONFIG["AWQ_OUTPUT_DIR"])
    target_dir.mkdir(parents=True, exist_ok=True)

    awq_recipe_cfg = CONFIG.get("AWQ_RECIPE", {})
    num_bits = int(awq_recipe_cfg.get("num_bits", 4))
    group_size = int(awq_recipe_cfg.get("group_size", 128))
    max_seq_len = int(awq_recipe_cfg.get("max_seq_len", 2048))
    zero_point = bool(awq_recipe_cfg.get("zero_point", True))

    def write_calibration_json(samples, directory: Path) -> Path:
        directory.mkdir(parents=True, exist_ok=True)
        file_path = directory / "calibration.json"
        payload = [{"text": sample} for sample in samples]
        file_path.write_text(
            "\n".join(json.dumps(item, ensure_ascii=False) for item in payload) + "\n",
            encoding="utf-8",
        )
        return file_path

    calib_source = CONFIG.get("AWQ_CALIBRATION_DATASET") or CONFIG.get("AWQ_CALIBRATION_SAMPLES")
    if calib_source is None:
        raise ValueError("AWQ_CALIBRATION_SAMPLES もしくは AWQ_CALIBRATION_DATASET を設定してください。")

    dataset_split = CONFIG.get("AWQ_CALIBRATION_DATASET_SPLIT", "train")
    if isinstance(calib_source, str):
        candidate_path = Path(calib_source)
        if candidate_path.exists():
            calib_data, text_column_hint = prepare_awq_calib_data(calib_source)
        else:
            try:
                from datasets import load_dataset

                dataset = load_dataset(calib_source, split=dataset_split)
                calib_data, text_column_hint = prepare_awq_calib_data(dataset)
            except Exception as exc:
                raise RuntimeError(f"キャリブレーションデータセット {calib_source} の読み込みに失敗しました: {exc}")
    else:
        calib_data, text_column_hint = prepare_awq_calib_data(calib_source)

    tokenizer_name_or_path = (
        CONFIG.get("TOKENIZER_NAME")
        or CONFIG.get("BASE_MODEL_NAME")
        or SOURCE_MODEL_PATH
    )
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name_or_path, trust_remote_code=True)

    model = AutoModelForCausalLM.from_pretrained(
        SOURCE_MODEL_PATH,
        device_map="auto",
        torch_dtype=torch.float16,
        trust_remote_code=True,
    )


    def summarise_samples(samples) -> None:
        if not isinstance(samples, list):
            return
        total_tokens = 0
        for idx, sample in enumerate(samples[:5]):
            token_count = len(tokenizer.encode(sample, add_special_tokens=False))
            total_tokens += token_count
            print(f"Sample {idx}: {token_count} token(s)")
        if len(samples) > 5:
            print(f"... (全 {len(samples)} 件)")
        print(f"合計トークン数: {total_tokens}")

    summarise_samples(calib_data)

    recipe = [
        AWQModifier(
            scheme="W4A16",
        ),
    ]



    oneshot_common_args = {
        "model": model,
        "tokenizer": tokenizer,
        "recipe": recipe,
        "max_seq_length": max_seq_len,
        "trust_remote_code_model": True,
    }


    if isinstance(calib_data, list):
        with tempfile.TemporaryDirectory() as tmpdir:
            dataset_dir = Path(tmpdir) / "dataset"
            write_calibration_json(calib_data, dataset_dir)
            run_oneshot(
                dataset="json",
                dataset_path=str(dataset_dir),
                text_column="text",
                num_calibration_samples=len(calib_data),
                **oneshot_common_args,
            )
    else:
        dataset_name = "json"
        dataset_path = None
        text_column = text_column_hint or CONFIG.get("AWQ_TEXT_FIELD", "text")
        if isinstance(calib_data, (str, Path)):
            candidate_path = Path(str(calib_data))
            if candidate_path.exists():
                dataset_path = str(candidate_path)
                if candidate_path.suffix.lower() == ".parquet":
                    dataset_name = "parquet"
                else:
                    dataset_name = "json"
            else:
                dataset_name = str(calib_data)
                dataset_path = None
        run_oneshot(
            dataset=dataset_name,
            dataset_path=dataset_path,
            text_column=text_column,
            num_calibration_samples=None,
            **oneshot_common_args,
        )

    target_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(target_dir, safe_tensors=True, save_compressed=True)
    tokenizer.save_pretrained(target_dir)

    AWQ_MODEL_PATH = str(target_dir)
    CONFIG["LAST_AWQ_DIR"] = AWQ_MODEL_PATH
    CONFIG["MODEL_PATH"] = AWQ_MODEL_PATH
    CONFIG["SOURCE_MODEL_PATH"] = AWQ_MODEL_PATH
    CONFIG["LAST_AWQ_RECIPE"] = {
        "num_bits": num_bits,
        "group_size": group_size,
        "max_seq_len": max_seq_len,
        "zero_point": zero_point,
    }
    persist_config(CONFIG)
    print("llm-compressor の oneshot で AWQ 量子化と保存が完了しました。出力先:", AWQ_MODEL_PATH)

print("AWQ モデルパス:", AWQ_MODEL_PATH)

In [ ]:
# === 5. 後片付け ===
import gc
import torch

_cleanup_targets = [name for name in ("SOURCE_MODEL_PATH", "AWQ_MODEL_PATH") if name in globals()]
print("保持中のパス変数:", {name: globals()[name] for name in _cleanup_targets})

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("GPU メモリを解放しました。次のステップへ進めます。")
